In [2]:
import numpy as np

encoded_sequences = np.load(
    "../encoded_sequences_len60_v2.npy"
)

print(encoded_sequences.shape)

(995, 62)


In [3]:
import torch

X = torch.tensor(
    encoded_sequences,
    dtype=torch.long
)

In [4]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False
)

In [5]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=62
    ):

        super().__init__()

        pe = torch.zeros(
            max_len,
            d_model
        )

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            )
            * (-math.log(10000.0) / d_model)
        )

        pe[:,0::2] = torch.sin(
            position * div_term
        )

        pe[:,1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self,x):

        return x + self.pe[:,:x.size(1)]

In [6]:
import torch
import torch.nn as nn

class TransformerVAE(nn.Module):

    def __init__(
        self,
        vocab_size=21,
        embed_dim=128,
        latent_dim=64,
        max_len=62
    ):

        super().__init__()

        self.max_len = max_len

        # Encoder

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.pos_encoder = PositionalEncoding(
            embed_dim,
            max_len
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=3
        )

        self.fc_mu = nn.Linear(
            embed_dim,
            latent_dim
        )

        self.fc_logvar = nn.Linear(
            embed_dim,
            latent_dim
        )

        # Decoder

        self.latent_to_embed = nn.Linear(
            latent_dim,
            embed_dim
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=embed_dim,
            nhead=8,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=3
        )

        self.output_layer = nn.Linear(
            embed_dim,
            vocab_size
        )

    def encode(self,x):

        x = self.embedding(x)

        x = self.pos_encoder(x)

        enc = self.encoder(x)

        pooled = enc.mean(dim=1)

        mu = self.fc_mu(pooled)

        logvar = self.fc_logvar(pooled)

        return mu, logvar

    def reparameterize(
        self,
        mu,
        logvar
    ):

        std = torch.exp(
            0.5 * logvar
        )

        eps = torch.randn_like(std)

        return mu + eps * std

    def decode(
        self,
        z,
        decoder_input
    ):

        tgt = self.embedding(
            decoder_input
        )

        tgt = self.pos_encoder(tgt)

        memory = self.latent_to_embed(
            z
        ).unsqueeze(1)

        out = self.decoder(
            tgt,
            memory
        )

        logits = self.output_layer(
            out
        )

        return logits

    def forward(self,x):

        mu, logvar = self.encode(x)

        z = self.reparameterize(
            mu,
            logvar
        )

        decoder_input = x[:,:-1]

        logits = self.decode(
            z,
            decoder_input
        )

        return logits, mu, logvar

In [7]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [11]:
vae = TransformerVAE(
    vocab_size=23,
    max_len=62
).to(device)

In [12]:
vae.load_state_dict(
    torch.load(
        "../final_model/transformer_vae_len60.pt",
        map_location=device
    )
)

vae.eval()

print("Loaded Successfully")

Loaded Successfully


In [14]:
vae.eval()

latent_list = []

with torch.no_grad():

    for batch in loader:

        x = batch[0].to(device)

        mu, logvar = vae.encode(x)

        latent_list.append(
            mu.cpu()
        )

latent_vectors_long = torch.cat(
    latent_list,
    dim=0
).numpy()

print(latent_vectors_long.shape)

(995, 64)


In [15]:
import numpy as np

np.save(
    "latent_vectors_len60_v2.npy",
    latent_vectors_long
)

print("Saved")

Saved


In [16]:
from sklearn.preprocessing import StandardScaler

scaler_long = StandardScaler()

latent_scaled_long = (
    scaler_long.fit_transform(
        latent_vectors_long
    )
)

print(
    latent_scaled_long.mean()
)

print(
    latent_scaled_long.std()
)

-2.3961666e-10
0.99999994


In [19]:
print(latent_vectors_long.shape)

print(latent_vectors_long.mean())
print(latent_vectors_long.std())

print(latent_scaled_long.mean())
print(latent_scaled_long.std())

(995, 64)
-0.013416984
0.13673699
-2.3961666e-10
0.99999994


In [20]:
import joblib
joblib.dump(
    scaler_long,
    "latent_scaler_len60_v2.pkl"
)

np.save(
    "latent_scaled_len60_v2.npy",
    latent_scaled_long
)